# Análise Exploratória Estratégica: Processamento de Petróleo no Brasil
**Objetivo:** Diagnosticar o comportamento volumétrico do refino nacional (*downstream*), mapeando a dependência de insumos (Petróleo Nacional vs. Importado), o market share das refinarias e a distribuição geográfica da capacidade de processamento entre 1990 e 2026.

## Etapa 1 — Entendimento do Problema
* **Domínio:** Infraestrutura de refino de petróleo e abastecimento nacional (Dados ANP).
* **Perguntas de Negócio:** 1. Qual é a evolução do market share de refino entre o insumo doméstico e o óleo importado ao longo da série histórica?
  2. Como a capacidade de processamento está concentrada regionalmente e por planta de refino?
  3. Quais choques macroeconômicos ou estruturais alteraram o ritmo de processamento de petróleo no país?
* **Métricas Principais (KPIs):** Volume Total Processado (m³), Share de Matéria-Prima (%), Crescimento Year-over-Year (YoY), Taxa de Utilização Relativa por Planta.
* **Limitações e Vieses:** O volume processado indica a carga total de entrada nas unidades de destilação primária, não representando diretamente a eficiência de conversão (rendimento de derivados nobres como diesel e gasolina) ou paradas programadas de manutenção técnica.

In [1]:
# =====================================================================
# CONFIGURAÇÃO DE AMBIENTE E BIBLIOTECAS
# =====================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from pathlib import Path
from scipy import stats
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Configurações de visualização
sns.set_theme(style="whitegrid")
pd.options.display.float_format = '{:,.2f}'.format

# Formatação para apresentação de casas decimais em padrão pt-BR
def fmt_br(x):
    """
    Formata valores numéricos para o padrão PT-BR (milhares com ponto, decimais com vírgula).
    """
    if pd.isna(x):
        return "0,00"
    return f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
# Para valores relativos (em %)
def fmt_pct_br(x):
    if pd.isna(x):
        return "0,00%"
    return f"{x:.2f}%".replace(".", ",")

# Carregamento dos dados
df_ref = pd.read_excel(Path.cwd().parent / "dataset" / "processamento_petroleo_brasil.xlsx")

## Etapa 2 — Diagnóstico e Padronização dos Dados
Nesta etapa, aplicamos o pipeline de limpeza de strings para o padrão `snake_case` e construímos a variável de tempo real (`datetime`).

In [2]:
# 1. Padronização para snake_case e eliminação de caracteres especiais nas colunas
df_ref.columns = (df_ref.columns
                  .str.lower()
                  .str.replace(' ', '_')
                  .str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8'))

# Renomeando colunas estratégicas para abreviação e facilidade de manipulação
df_ref = df_ref.rename(columns={'unidade_da_federacao': 'uf'})

# 2. Mapeamento uniforme de meses (idêntico aos projetos anteriores)
meses_map = {
    'JAN': '01', 'FEV': '02', 'MAR': '03', 'ABR': '04', 
    'MAI': '05', 'JUN': '06', 'JUL': '07', 'AGO': '08', 
    'SET': '09', 'OUT': '10', 'NOV': '11', 'DEZ': '12'
}
df_ref['mes_num'] = df_ref['mes'].str.upper().map(meses_map)

# Removendo os meses do ano de 2026 (incompleto)
df_ref = df_ref.loc[df_ref['ano'] != 2026].copy()

# 3. Geração do vetor de data verdadeira (Datetime)
df_ref['data'] = pd.to_datetime(df_ref['ano'].astype(str) + '-' + df_ref['mes_num'] + '-01')

# 4. Ajuste de tipagem volumétrica e tratamento de registros nulos
df_ref['processado'] = pd.to_numeric(df_ref['processado'], errors='coerce').fillna(0)

display(HTML(f"<b style='color:#00A8E8;'>✓ Código Executado. Dataset organizado!</b> Total de Linhas: <b>{df_ref.shape[0]:,}</b> | Total de Colunas <b>{df_ref.shape[1]}</b>."))
display(df_ref.head(5))

,ano,mes,uf,refinaria,materia_prima,processado,mes_num,data
0,1990,DEZ,BAHIA,DAX OIL,OUTRAS CARGAS,0,12,1990-12-01
1,1990,SET,BAHIA,DAX OIL,PETRÓLEO IMPORTADO,0,09,1990-09-01
2,1990,OUT,BAHIA,DAX OIL,PETRÓLEO IMPORTADO,0,10,1990-10-01
3,1990,NOV,BAHIA,DAX OIL,PETRÓLEO IMPORTADO,0,11,1990-11-01
4,1990,DEZ,BAHIA,DAX OIL,PETRÓLEO IMPORTADO,0,12,1990-12-01


In [ ]:
# # Exportando o dataset tratado e filtrado para a pronta aplicação no dash
# nome_arquivo_saida = Path.cwd().parent / "projeto_downstream" / "processamento_petroleo_filtered.parquet"
# df.to_parquet(nome_arquivo_saida, index=False)

# print(f"Sucesso! Dataset exportado e pronto para o Dash: {nome_arquivo_saida}")

## Etapa 3 — Perfil Estatístico Profundo da Carga de Refino
Análise das propriedades estatísticas do volume processado (m³) para quantificar a dispersão e assimetria do ecossistema de refino brasileiro.

In [3]:
# Estatística descritiva do volume de processamento
stats_prod = df_ref['processado'].describe(percentiles=[.25, .5, .75, .90, .99]).apply(fmt_br)
assimetria = df_ref['processado'].skew()
curtose = df_ref['processado'].kurtosis()

print("\n" + "="*50)
print("--- 📊 PERFIL ESTATÍSTICO DE PRODUÇÃO (m³) 📊 ---")
print("="*50)
print(stats_prod)

print("\n" + "="*50)
print(f"Assimetria (Skewness): {fmt_br(assimetria)} -> Uma assimetria alta positiva indica cauda longa à direita (poucos registros com volumes astronômicos).")
print(f"Curtose (Kurtosis): {fmt_br(curtose)} -> Dados concentrados em torno de valores baixos/médios, com picos extremos distantes da média.")


--- 📊 PERFIL ESTATÍSTICO DE PRODUÇÃO (m³) 📊 ---
count       22.939,00
mean       153.066,86
std        296.475,54
min              0,00
25%              0,00
50%          8.964,00
75%        146.253,00
90%        595.546,80
99%      1.333.150,48
max      2.035.933,00
Name: processado, dtype: str

Assimetria (Skewness): 2,51 -> Uma assimetria alta positiva indica cauda longa à direita (poucos registros com volumes astronômicos).
Curtose (Kurtosis): 6,75 -> Dados concentrados em torno de valores baixos/médios, com picos extremos distantes da média.


## Etapa 4 — Segmentação e Visualização Estratégica
Exploração do comportamento de refino através de três camadas gráficas balanceadas:
1. **Univariada/Simples:** Participação de mercado por tipo de matéria-prima.
2. **Bivariada/Série Temporal:** Evolução temporal do perfil de suprimento da carga de refino.
3. **Multivariada/Complexa:** Concentração de volume processado por Unidade da Federação e Planta Industrial.

In [4]:
# Participação Geral do Processamento (DONUT CHART)
df_rosca = df_ref.groupby("materia_prima")["processado"].sum().reset_index()

fig_rosca = px.pie(
    df_rosca, 
    values='processado', 
    names='materia_prima', 
    hole=0.55,
    title='Participação Geral do Processamento por Tipo de Carga',
    template='plotly_dark',
    color='materia_prima',
    color_discrete_sequence=["#FBC28A", "#F38370", "#831E70"]
)

# OTIMIZAÇÃO DE UX: Rótulos posicionados externamente com linhas de chamada (evita sobreposição)
fig_rosca.update_traces(
    textposition='outside',
    textinfo='percent+label',
    hovertemplate=
    "<span style='color:white'><b>Matéria Prima:</b> %{label}<br><b>Volume Processado:</b> %{value:,.2f} m³<extra></extra>"
)

fig_rosca.update_layout(
    separators=",.",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    legend=dict(orientation="h", yanchor="bottom", y=-0.19, xanchor="right", x=1.12)
)

fig_rosca.show()

In [5]:
# 1. Market Share Histórico por Matéria-Prima
share_mat = (
    df_ref
    .groupby('materia_prima', as_index=False)['processado']
    .sum()
)

share_mat['share_%'] = (
    share_mat['processado'] /
    share_mat['processado'].sum()
) * 100

share_mat = share_mat.sort_values(
    'share_%',
    ascending=False
)

share_mat["processado_br"] = share_mat["processado"].apply(fmt_br)
share_mat["share_br"] = share_mat["share_%"].apply(fmt_pct_br)

fig_share_mat = px.bar(
    share_mat,
    x='share_%',
    y='materia_prima',
    orientation='h',
    custom_data=['processado_br', 'share_br'],
    color='materia_prima',
    color_discrete_sequence=[
        "#831E70",
        "#F38370",
        "#FBC28A"
    ],
    title='Market Share Histórico da Carga de Refino por Matéria-Prima (%)',
    template='plotly_dark',
    labels={
        "materia_prima": "Matéria-Prima"
    }
)

fig_share_mat.update_layout(
    separators=",.",
    xaxis_title='Participação Acumulada (%)',
    yaxis_title='',
    showlegend=False
)

fig_share_mat.update_traces(
    hovertemplate=
    "<b>Matéria-Prima:</b> %{y}<br>" +
    "<b>Volume Processado:</b> %{customdata[0]} m³<br>" +
    "<b>Participação:</b> %{customdata[1]}" +
    "<extra></extra>"
)

fig_share_mat.show()

In [6]:
# 2. Evolução Histórica de Processamento por Origem do Óleo
df_temp_mat = (
    df_ref
    .groupby(['data', 'materia_prima'], as_index=False)['processado']
    .sum()
)

fig_serie = px.line(
    df_temp_mat,
    x='data',
    y='processado',
    color='materia_prima',
    category_orders={
        "materia_prima": [
            "PETRÓLEO NACIONAL",
            "PETRÓLEO IMPORTADO",
            "OUTRAS CARGAS"
        ]
    },
    color_discrete_sequence=[
        "#831E70",
        "#F38370",
        "#FBC28A"
    ],
    title='Evolução de Carga Processada nas Refinarias: Nacional vs. Importado',
    template='plotly_dark',
    labels={
        "materia_prima": "Matéria-Prima: "
    }
)

fig_serie.update_layout(
    separators=",.",
    xaxis_title="Ano",
    yaxis_title="Volume Refinado (m³)",
    hovermode="x unified"
)

fig_serie.update_traces(
    line=dict(width=3),
    hovertemplate=
    "<b>Matéria-Prima:</b> %{fullData.name}<br>" +
    "<b>Volume Processado:</b> %{y:,.1f} m³" +
    "<extra></extra>"
)

fig_serie.show()

In [7]:
# 3. Heatmap: UF x Matéria-Prima (% do Total Nacional)
pivot_uf_ref = (
    df_ref
    .pivot_table(
        index='uf',
        columns='materia_prima',
        values='processado',
        aggfunc='sum'
    )
    .fillna(0)
)

pivot_uf_ref_pct = (
    pivot_uf_ref /
    pivot_uf_ref.sum().sum()
) * 100

fig_heatmap = go.Figure(
    data=go.Heatmap(
        z=pivot_uf_ref_pct.values,
        x=pivot_uf_ref_pct.columns,
        y=pivot_uf_ref_pct.index,
        colorscale=[
            [0, "#FBC28A"],
            [0.25, "#F38370"],
            [0.50, "#831E70"],
            [1, "#2D1E3E"]
        ],
        hovertemplate=
        "<b>UF:</b> %{y}<br>" +
        "<b>Matéria-Prima:</b> %{x}<br>" +
        "<b>Participação:</b> %{z:,.2f} %<extra></extra>"
    )
)

fig_heatmap.update_layout(
    separators=",.",
    title='Heatmap de Concentração Industrial (% de Carga sobre o Total Nacional Geral)',
    template='plotly_dark',
    xaxis_title='Tipo de Matéria-Prima',
    yaxis_title='Estado (UF)'
)

fig_heatmap.show()

In [10]:
df_mix = df_ref.groupby(['uf', 'materia_prima'])['processado'].sum().reset_index()

df_mix["processado_br"] = df_mix["processado"].apply(fmt_br)

# Ordenando o eixo Y pelos estados com maior volume total
ordem_estados = df_mix.groupby('uf')['processado'].sum().sort_values(ascending=False).index

fig_bar = px.bar(
    df_mix,
    y='uf',
    x='processado',
    color='materia_prima',
    custom_data=["processado_br"],
    color_discrete_sequence=["#FBC28A", "#F38370", "#831E70"],
    title="Mix de Dependência de Matéria-Prima por Estado",
    labels={"materia_prima": "Matéria Prima: "},
    barmode='stack',
    category_orders={"uf": ordem_estados}
)

fig_bar.update_layout(
    template="plotly_dark", 
    yaxis_title="Estado", 
    xaxis_title="Volume Total Processado (m³)"
)

fig_bar.update_traces(
    hovertemplate=
    "<span style='color:white'><b>Matéria Prima:</b> %{fullData.name}</span><br>" +
    "<span style='color:white'><b>Processado:</b> %{customdata[0]} m³</span>" +
    "<extra></extra>"
)

fig_bar.show()

##### Concentração Geográfica e Market Share (Treemap)
Mapeamento hierárquico do risco de concentração de capacidade. Identificação clara dos polos dominantes (Sudeste) e o peso individual de cada refinaria dentro de sua respectiva UF.

In [12]:
# Calculando a volumetria e removendo zeros para otimizar o algoritmo do Treemap
df_share = (
    df_ref.groupby(['uf', 'refinaria'])['processado']
      .sum()
      .reset_index()
)

df_share = df_share[df_share['processado'] > 0]

# Coluna formatada para PT-BR
df_share["processado_br"] = (
    df_share["processado"]
    .apply(fmt_br)
)

fig_tree = px.treemap(
    df_share,
    path=[px.Constant("Brasil"), 'uf', 'refinaria'],
    values='processado',
    custom_data=['processado_br'],
    title="Mapa de Concentração do Refino Nacional (Market Share)",
    color='processado',
    color_continuous_scale='Sunsetdark',
    template='plotly_dark'
)

fig_tree.update_traces(
    root_color="lightgrey",
    texttemplate="%{label}<br>%{value:,.0f}",
    hovertemplate=
    "<b>%{label}</b><br>" +
    "<b>Processado:</b> %{customdata[0]} m³<br>" +
    "<b>Participação:</b> %{percentParent:.2%}" +
    "<extra></extra>"
)

fig_tree.update_layout(
    margin=dict(t=50, l=25, r=25, b=25),
    hoverlabel=dict(
        bgcolor="rgba(0,0,0,0.9)",
        font_color="white",
        font_size=13
    ),
    coloraxis_colorbar=dict(
        title="Volume Processado"
    )
)

fig_tree.show()

## Etapa 5 — Análise de Ritmo Temporal (Crescimento YoY do Refino)
Avaliação da velocidade de expansão ou contração do refino nacional ao longo das décadas para capturar a resposta da infraestrutura a choques econômicos.

In [13]:
# Crescimento YoY do Refino de Petróleo
ref_anual = (
    df_ref
    .groupby('ano', as_index=False)['processado']
    .sum()
)

ref_anual['yoy_growth_%'] = (
    ref_anual['processado']
    .pct_change() * 100
)

ref_anual = ref_anual.dropna()

ref_anual['cor'] = np.where(
    ref_anual['yoy_growth_%'] >= 0,
    '#09703b',
    '#D72631'
)

fig_yoy = px.bar(
    ref_anual,
    x='ano',
    y='yoy_growth_%',
    color='cor',
    color_discrete_map='identity',
    title='Variação Percentual Ano contra Ano (YoY %) do Refino de Petróleo'
)

fig_yoy.update_traces(
    textposition='outside',
    texttemplate='%{y:.1f}%',
    hovertemplate=
    '<b>Ano:</b> %{x}<br>' +
    '<b>Crescimento:</b> %{y:,.2f}<extra></extra>'
)

fig_yoy.update_layout(
    separators=",.",
    template='plotly_dark',
    title=dict(
        x=0.5,
        font=dict(
            size=22,
            family='Arial Black'
        )
    ),
    xaxis_title='Ano',
    yaxis_title='Crescimento (%)',
    xaxis=dict(
        tickmode='linear',
        dtick=1
    ),
    yaxis=dict(
        ticksuffix='%'
    ),
    font=dict(
        family='Arial',
        size=14
    ),
    showlegend=False
)

fig_yoy.show()

## Etapa 6 — Dossiê Estratégico do Downstream Industrial

### Resumo Executivo
1. **A Consolidação da Autossuficiência de Insumo:** A análise temporal exibe a inversão estrutural histórica da matriz de refino brasileira. A dependência crônica de *Petróleo Importado* nas décadas de 90 foi progressivamente substituída por *Petróleo Nacional*, acompanhando o aumento da capacidade de extração marítima (Upstream).
2. **Gargalo de Infraestrutura Geográfica:** O parque de refino apresenta uma forte inércia espacial focada no Sudeste. Isto expõe um descasamento logístico regional natural, onde estados de alta tração de consumo (como a fronteira agrícola do Centro-Oeste) não possuem infraestrutura de refino proporcional, exigindo longos deslocamentos via cabotagem ou dutos.

### Hipóteses de Negócio para Integração Macro (Produção $\rightarrow$ Refino $\rightarrow$ Vendas)
* **Hipótese 1:** "Embora o volume de Petróleo Nacional processado tenha crescido, o rendimento final de Óleo Diesel nas refinarias não acompanhou a velocidade do aumento das vendas de combustíveis no interior do país, forçando um aumento paradoxal na importação de derivados refinados."
  * *Como Validar:* Realizar o merge entre este dataframe e o de *Vendas de Combustíveis* pela chave `data`. Calcular a razão entre o volume total de petróleo processado contra o volume total de diesel vendido por período.
* **Hipótese 2:** "Períodos de contração acentuada (YoY negativa) na carga de refino nacional correlacionam-se diretamente com o aumento imediato no volume de vendas de combustíveis importados pelas distribuidoras de capital privado."
  * *Como Validar:* Mapear meses com quebras estruturais ou paradas programadas de refino e testar o coeficiente de correlação com as vendas de derivados de aviação e automotivos nas UFs portuárias.

### Recomendações Estratégicas (Actionables)
* **Otimização do Perfil de Carga:** As refinarias originalmente projetadas para processar óleo leve importado precisam manter o cronograma de investimentos em modernização tecnológica (unidades de hidrotratamento e craqueamento) para assimilar a crescente oferta de petróleo nacional pesado/médio vindo do Pré-Sal, otimizando as margens operacionais do setor (*refining margins*).